# Width sweeps on Google Colab

Colab twin of `kaggle/width_sweep.ipynb` for when the Kaggle GPU quota is used up.
**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**; keep the tab open.

Colab can disconnect at any time, so everything that matters lives on Google Drive
(`MyDrive/mpgnn/`):
- `results/width/` — every finished run is appended there as it completes;
- `width_cache/` — the datasets, built once (~3 h on Colab's 2 CPUs) and reused.

After a disconnect just **Run all** again: datasets load from Drive and the sweeps skip
runs already on disk (only the run in progress at the disconnect is repeated).
Download `MyDrive/mpgnn/results/width/` when done.

In [ ]:
SWEEPS = ["enc_khop2", "enc_khop4", "enc_rwse8", "enc_rwse16"]   # sweep keys of width_sweep.py
SHARDS = 1          # workers per sweep; 1 T4 + 2 CPUs: ~4 workers in total is plenty
BRANCH = "master"

DRIVE = "/content/drive/MyDrive/mpgnn"
import os
os.environ["WIDTH_CACHE"] = "/content/width_cache"   # local disk: fast, and atomic renames work

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE}/results/width", exist_ok=True)
os.makedirs(f"{DRIVE}/width_cache", exist_ok=True)

%cd /content
!rm -rf mpgnn && git clone -q --depth 1 -b $BRANCH https://github.com/barsboldb/mpgnn.git
%cd /content/mpgnn
!pip install -q torch-geometric
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git log -1 --format='%h %s'

In [ ]:
# Datasets: copy what Drive already has, build the rest, save new ones back to Drive.
!mkdir -p /content/width_cache && cp -n $DRIVE/width_cache/*.pt /content/width_cache/ 2>/dev/null; ls /content/width_cache | wc -l
for sw in SWEEPS:
    !python -u width_sweep.py {sw} --prepare
!cp -n /content/width_cache/*.pt $DRIVE/width_cache/ && ls -lh $DRIVE/width_cache | tail -n +2 | wc -l

In [ ]:
# Results go straight to Drive: the repo's results/width becomes a link to it.
import subprocess, time, torch
!rm -rf results/width && ln -s $DRIVE/results/width results/width
out = "results/width"
procs = []
for sw in SWEEPS:
    for i in range(SHARDS):
        log = open(f"{out}/{sw}.shard{i}.log", "a")
        procs.append(subprocess.Popen(["python", "-u", "width_sweep.py", sw, "--shard", f"{i}/{SHARDS}"],
                                      stdout=log, stderr=subprocess.STDOUT))
print(f"{len(procs)} jobs on 1 GPU")
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(120)
    done = sum(1 for f in os.listdir(out) if f.endswith(".jsonl") and f.split(".")[0] in SWEEPS
               for _ in open(f"{out}/{f}"))
    print(f"{(time.time() - t0) / 60:5.1f} min  runs saved: {done}", flush=True)
print("exit codes:", [p.returncode for p in procs])

In [ ]:
for sw in SWEEPS:
    !python width_sweep.py {sw} --analyze